# Решения: CLI-практика

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

In [ ]:
script = Path('train_cli.py')
data_path = Path('../../data/bank_marketing_slim.csv')
cmd = [
    sys.executable,
    str(script),
    '--data',
    str(data_path),
    '--threshold',
    '0.45',
]
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
metrics = json.loads(proc.stdout)
ok = bool(
    (not metrics['duration_in_features'])
    and 0.0 <= metrics['accuracy'] <= 1.0
    and 0.0 <= metrics['precision'] <= 1.0
    and 0.0 <= metrics['recall'] <= 1.0
    and 0.0 <= metrics['f1'] <= 1.0
)
results = {}
for thr in (0.35, 0.55):
    cmd_thr = [sys.executable, str(script), '--data', str(data_path), '--threshold', str(thr)]
    proc_thr = subprocess.run(cmd_thr, capture_output=True, text=True, check=False)
    if proc_thr.returncode != 0:
        raise RuntimeError(proc_thr.stderr or proc_thr.stdout)
    results[thr] = json.loads(proc_thr.stdout)
better_thr = 0.35 if results[0.35]['f1'] >= results[0.55]['f1'] else 0.55
CLI_NOTE = (
    f'CLI-проверка показала, что threshold={better_thr:.2f} даёт лучший F1 на test. '
    'Команда запуска воспроизводима, а запрет duration зафиксирован в самом скрипте через assert.'
)
print(metrics)
print(results)
print('CLI_OK=', ok)
print(CLI_NOTE)